In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Ensure the notebook can find the /src directory
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

In [2]:
from src.data_loader import VesselDataLoader
from src.visualizer import plot_block_space, plot_mode_statistics
from src.data_processing import engineer_telemetry_features
from src.mission_profiler import MissionProfiler

In [3]:
filename1 = "Rotherhithe_voy_179.csv"
filename2 = "Wembley_voy_236.csv"

filename = filename1 

raw_data_file = root_path / "data" / "raw" / filename

loader = VesselDataLoader(raw_data_file)
raw_df = loader.load_and_clean()

processed_df = engineer_telemetry_features(raw_df, filter_method='savgol')

profiler = MissionProfiler(speed_threshold = 0.01)

modes_df = profiler.classify_modes(processed_df)
global_stats = profiler.extract_global_statistics(modes_df)

registry_df = profiler.generate_block_registry(modes_df, source_file_name=filename, merge_loitering=False, unify_port_ops=False)
display(registry_df.head())

,Source_File,Start_Time,Duration_h,Energy_kWh,Mean_Power_kW,H2_Rate_Lower_kg_h,H2_Rate_Upper_kg_h,Fatigue_Damage_Rate,Mean_Power_Fluctuation_Intensity,Stay_ID,MODE,Loitering_Handling,Port_Handling
0,Rotherhithe_voy_179.csv,2025-12-23 14:35,72.250000,29442.451376,407.507978,22.236603,27.178070,6643.580125,6.268572,2,Sea_Loitering,Separated,NaN
1,Rotherhithe_voy_179.csv,2025-12-22 10:50,31.416667,13556.667324,431.511957,23.546434,28.778975,2651.621846,2.731760,2,Sea_Transit_Ballast,Separated,NaN
2,Rotherhithe_voy_179.csv,2025-12-27 15:55,5.250000,2414.215185,459.850511,25.092792,30.668968,2001.526600,2.184905,3,Port_Idle,NaN,Separated
3,Rotherhithe_voy_179.csv,2025-12-26 18:30,21.416667,10237.224212,478.002687,26.083307,31.879598,4604.059727,4.363502,3,Port_Loading,NaN,Separated
4,Rotherhithe_voy_179.csv,2025-12-30 00:55,7.083333,3535.454841,499.123036,27.235787,33.288184,1747.061786,1.801520,4,Sea_Loitering,Separated,NaN


In [11]:
fig_bricks_fluctuation = plot_block_space(registry_df, y_axis_metric='Mean_Power_Fluctuation_Intensity')
fig_bricks_fluctuation.show()
fig_bricks_fatigue = plot_block_space(registry_df, y_axis_metric='Fatigue_Damage_Rate')
fig_bricks_fatigue.show()


In [12]:
fig_stats_fluctuation = plot_mode_statistics(global_stats, y_axis_metric='Mean_Power_Fluctuation_Intensity')
fig_stats_fluctuation.show()

fig_stats_fatigue = plot_mode_statistics(global_stats, y_axis_metric='Fatigue_Damage_Rate')
fig_stats_fatigue.show()